<a href="https://colab.research.google.com/github/riidhooo12-oss/Pemograman-Berbasis-Objek/blob/main/Ridho_Arsyil_Hakim_4_33_25_1_23_PBO_JOBSHEET_11_INTEGRASI_OOP_APLIKASI_PENGELUARAN_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#JOBSHEET 11: INTEGRASI OOP DALAM APLIKASI PENGELUARAN SEDERHANA



---

PEMROGRAMAN BERORIENTASI OBJEK


```
Nama: Ridho Arsyil Hakim
NIM: 4.33.25.1.23
KELAS: TI-1B
```

### Langkah 1: Persiapan Awal (Konfigurasi & Setup Database)

In [ ]:
#LANGKAH 1 - FILE 1: konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Hiburan", "Tagihan",
                        "Belanja", "Kesehatan", "Pendidikan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

print(f"DB_PATH       : {DB_PATH}")
print(f"Kategori      : {KATEGORI_PENGELUARAN}")
print(f"Default       : {KATEGORI_DEFAULT}")

DB_PATH       : /content/pengeluaran_harian.db
Kategori      : ['Makanan', 'Transportasi', 'Hiburan', 'Tagihan', 'Belanja', 'Kesehatan', 'Pendidikan', 'Lainnya']
Default       : Lainnya


In [ ]:
#LANGKAH 1 - FILE 2: setup_db_pengeluaran.py
import sqlite3
import os

def setup_database(db_path=DB_PATH):
    print(f"Memeriksa/membuat database di: {db_path}")
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        print(" Membuat tabel 'transaksi' (jika belum ada)...")
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f" -> Error SQLite saat setup: {e}")
        return False
    finally:
        if conn:
            conn.close()
            print(" -> Koneksi DB setup ditutup.")

if __name__ == "__main__":
    print("--- Memulai Setup Database Pengeluaran ---")
    if setup_database():
        print(f"\nSetup database '{os.path.basename(DB_PATH)}' selesai.")
    else:
        print(f"\nSetup database GAGAL.")
    print("--- Setup Database Selesai ---")

# Jalankan setup
print("--- Memulai Setup Database Pengeluaran ---")
if setup_database():
    print(f"\nSetup database '{os.path.basename(DB_PATH)}' selesai.")
else:
    print(f"\nSetup database GAGAL.")
print("--- Setup Database Selesai ---")

--- Memulai Setup Database Pengeluaran ---
Memeriksa/membuat database di: /content/pengeluaran_harian.db
 Membuat tabel 'transaksi' (jika belum ada)...
 -> Tabel 'transaksi' siap.
 -> Koneksi DB setup ditutup.

Setup database 'pengeluaran_harian.db' selesai.
--- Setup Database Selesai ---
--- Memulai Setup Database Pengeluaran ---
Memeriksa/membuat database di: /content/pengeluaran_harian.db
 Membuat tabel 'transaksi' (jika belum ada)...
 -> Tabel 'transaksi' siap.
 -> Koneksi DB setup ditutup.

Setup database 'pengeluaran_harian.db' selesai.
--- Setup Database Selesai ---


### Langkah 2: Modul Akses Database (database.py)

In [ ]:
#LANGKAH 2: database.py
import sqlite3
import pandas as pd

def get_db_connection(db_path=DB_PATH):
    """Membuka dan mengembalikan koneksi baru ke database SQLite."""
    try:
        conn = sqlite3.connect(db_path, timeout=10,
                               detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row  # Akses kolom by name
        return conn
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Koneksi DB gagal: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    """Menjalankan query non-SELECT. Mengembalikan lastrowid jika INSERT."""
    conn = get_db_connection()
    if not conn:
        return None
    last_id = None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        conn.commit()
        last_id = cursor.lastrowid
        return last_id
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Query gagal: {e} | Query: {query[:60]}")
        conn.rollback()
        return None
    finally:
        if conn:
            conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    """Menjalankan query SELECT dan mengembalikan hasil."""
    conn = get_db_connection()
    if not conn:
        return None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
        return result
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Fetch gagal: {e} | Query: {query[:60]}")
        return None
    finally:
        if conn:
            conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    """Menjalankan query SELECT dan mengembalikan DataFrame Pandas."""
    conn = get_db_connection()
    if not conn:
        return pd.DataFrame()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e:
        print(f"ERROR [database.py] Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        if conn:
            conn.close()

def setup_database_initial():
    """Memastikan tabel transaksi ada (dipanggil oleh AnggaranHarian jika perlu)."""
    print(f"Memeriksa/membuat tabel di database (via database.py): {DB_PATH}")
    conn = get_db_connection()
    if not conn:
        return False
    try:
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT, deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0), kategori TEXT,
            tanggal DATE NOT NULL );"""
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f"Error SQLite saat setup tabel: {e}")
        return False
    finally:
        if conn:
            conn.close()

print("[OK] Modul database.py siap.")

[OK] Modul database.py siap.


### Langkah 3: Modul Model Data (model.py)

In [ ]:
#LANGKAH 3: model.py
import datetime

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran (Data Class)."""
    def __init__(self, deskripsi: str, jumlah: float, kategori: str,
                 tanggal, id_transaksi=None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
            if jumlah_float <= 0:
                print(f"Peringatan: Jumlah '{jumlah}' harus positif.")
        except (ValueError, TypeError):
            self.jumlah = 0.0
            print(f"Peringatan: Jumlah '{jumlah}' tidak valid.")
        self.kategori = str(kategori) if kategori else "Lainnya"
        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
                print(f"Peringatan: Format tgl '{tanggal}' salah.")
        else:
            self.tanggal = datetime.date.today()
            print(f"Peringatan: Tipe tgl '{type(tanggal)}' tidak valid.")

    def __repr__(self) -> str:
        try:
            import locale
            locale.setlocale(locale.LC_ALL, 'id_ID.UTF8')
            jml_str = locale.format_string("%.0f", self.jumlah, grouping=True)
        except:
            jml_str = f"{self.jumlah:.0f}"
        return (f"Transaksi(ID:{self.id}, Tgl:{self.tanggal.strftime('%Y-%m-%d')}, "
                f"Jml:{jml_str}, Kat:'{self.kategori}', Desc:'{self.deskripsi}')")

    def to_dict(self) -> dict:
        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

# --- Uji kelas Transaksi ---
if __name__ == "__main__":
    tx_test = Transaksi("Makan siang", 25000, "Makanan", datetime.date.today())
    print(f"Objek Transaksi : {tx_test}")
    print(f"to_dict()       : {tx_test.to_dict()}")

tx_test = Transaksi("Makan siang", 25000, "Makanan", datetime.date.today())
print(f"Objek Transaksi : {tx_test}")
print(f"to_dict()       : {tx_test.to_dict()}")

Objek Transaksi : Transaksi(ID:None, Tgl:2026-06-05, Jml:25000, Kat:'Makanan', Desc:'Makan siang')
to_dict()       : {'deskripsi': 'Makan siang', 'jumlah': 25000.0, 'kategori': 'Makanan', 'tanggal': '2026-06-05'}
Objek Transaksi : Transaksi(ID:None, Tgl:2026-06-05, Jml:25000, Kat:'Makanan', Desc:'Makan siang')
to_dict()       : {'deskripsi': 'Makan siang', 'jumlah': 25000.0, 'kategori': 'Makanan', 'tanggal': '2026-06-05'}


### Langkah 4: Modul Manajer Anggaran (manajer_anggaran.py)

In [ ]:
#LANGKAH 4: manajer_anggaran.py
import datetime
import pandas as pd

class AnggaranHarian:
    """Mengelola logika bisnis pengeluaran harian (Repository Pattern)."""
    _db_setup_done = False  # Flag untuk memastikan setup DB hanya dicek sekali per sesi

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            print("[AnggaranHarian] Melakukan pengecekan/setup database awal...")
            if setup_database_initial():  # Panggil fungsi setup dari database.py
                AnggaranHarian._db_setup_done = True
                print("[AnggaranHarian] Database siap.")
            else:
                print("[AnggaranHarian] KRITICAL: Setup database awal GAGAL!")

    def tambah_transaksi(self, transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0:
            return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (
            transaksi.deskripsi,
            transaksi.jumlah,
            transaksi.kategori,
            transaksi.tanggal.strftime("%Y-%m-%d")
        )
        last_id = execute_query(sql, params)
        if last_id is not None:
            transaksi.id = last_id
            return True
        return False

    def get_semua_transaksi_obj(self) -> list:
        sql = "SELECT id, deskripsi, jumlah, kategori, tanggal FROM transaksi ORDER BY tanggal DESC, id DESC"
        rows = fetch_query(sql, fetch_all=True)
        transaksi_list = []
        if rows:
            for row in rows:
                transaksi_list.append(Transaksi(
                    id_transaksi=row['id'],
                    deskripsi=row['deskripsi'],
                    jumlah=row['jumlah'],
                    kategori=row['kategori'],
                    tanggal=row['tanggal']
                ))
        return transaksi_list

    def get_dataframe_transaksi(self, filter_tanggal=None) -> pd.DataFrame:
        query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        if filter_tanggal:
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"
        df = get_dataframe(query, params=params)
        if not df.empty:
            try:
                import locale
                locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
                df['Jumlah (Rp)'] = df['jumlah'].map(
                    lambda x: locale.currency(x or 0, grouping=True, symbol='Rp ')[:-3]
                )
            except:
                df['Jumlah (Rp)'] = df['jumlah'].map(
                    lambda x: f"Rp {x or 0:,.0f}".replace(",", ".")
                )
            df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]
        return df

    def hitung_total_pengeluaran(self, tanggal=None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        if tanggal:
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        result = fetch_query(sql, params=params, fetch_all=False)
        if result and result[0] is not None:
            return float(result[0])
        return 0.0

    def get_pengeluaran_per_kategori(self, tanggal=None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        if tanggal:
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"
        rows = fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else "Lainnya"
                jumlah = float(row[1]) if row[1] is not None else 0.0
                hasil[kategori] = jumlah
        return hasil

print("[OK] Kelas AnggaranHarian siap.")

[OK] Kelas AnggaranHarian siap.


### Langkah 5: Aplikasi Utama Streamlit (main_app.py)

In [ ]:
#LANGKAH 5: main_app.py
# Catatan: Kode Streamlit tidak dapat dijalankan langsung dalam Jupyter/Colab.
# Simpan sebagai main_app.py dan jalankan dengan: streamlit run main_app.py

streamlit_code = """
# main_app.py
import streamlit as st
import datetime
import pandas as pd
import locale

try:
    locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
except locale.Error:
    try:
        locale.setlocale(locale.LC_ALL, 'Indonesian_Indonesia.1252')
    except:
        print("Locale id_ID/Indonesian tidak tersedia.")

def format_rp(angka):
    try:
        return locale.currency(angka or 0, grouping=True, symbol='Rp ')[:-3]
    except:
        return f"Rp {angka or 0:,.0f}".replace(",", ".")

try:
    from model import Transaksi
    from manajer_anggaran import AnggaranHarian
    from konfigurasi import KATEGORI_PENGELUARAN
except ImportError as e:
    st.error(f"Gagal mengimpor modul: {e}. Pastikan file .py lain ada.")
    st.stop()

st.set_page_config(page_title="Catatan Pengeluaran", layout="wide",
                   initial_sidebar_state="expanded")

@st.cache_resource
def get_anggaran_manager():
    print(">>> STREAMLIT: (Cache Resource) Menginisialisasi AnggaranHarian...")
    return AnggaranHarian()

anggaran = get_anggaran_manager()

def halaman_input(anggaran):
    st.header("\U0001f4b8 Tambah Pengeluaran Baru")
    with st.form("form_transaksi_baru", clear_on_submit=True):
        col1, col2 = st.columns([3, 1])
        with col1: deskripsi = st.text_input("Deskripsi*", placeholder="Contoh: Makan siang")
        with col2: kategori = st.selectbox("Kategori*:", KATEGORI_PENGELUARAN, index=0)
        col3, col4 = st.columns([1, 1])
        with col3: jumlah = st.number_input("Jumlah (Rp)*:", min_value=0.01, step=1000.0,
                                            format="%.0f", value=None, placeholder="Contoh: 25000")
        with col4: tanggal = st.date_input("Tanggal*:", value=datetime.date.today())
        submitted = st.form_submit_button("\U0001f4be Simpan Transaksi")
        if submitted:
            if not deskripsi: st.warning("Deskripsi wajib diisi!", icon="\u26a0\ufe0f")
            elif jumlah is None or jumlah <= 0: st.warning("Jumlah wajib diisi!", icon="\u26a0\ufe0f")
            else:
                with st.spinner("Menyimpan..."):
                    tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                    if anggaran.tambah_transaksi(tx):
                        st.success("Transaksi berhasil disimpan!", icon="\u2705")
                        st.cache_data.clear(); st.rerun()
                    else: st.error("Gagal menyimpan transaksi.", icon="\u274c")

def halaman_riwayat(anggaran):
    st.subheader("\U0001f4cb Detail Semua Transaksi")
    if st.button("\U0001f504 Refresh Riwayat"): st.cache_data.clear(); st.rerun()
    with st.spinner("Memuat riwayat..."): df_transaksi = anggaran.get_dataframe_transaksi()
    if df_transaksi is None: st.error("Gagal mengambil riwayat transaksi.")
    elif df_transaksi.empty: st.info("Belum ada transaksi yang tercatat.")
    else:
        st.dataframe(df_transaksi, use_container_width=True, hide_index=True)
        st.divider()
        st.subheader("\U0001f5d1\ufe0f Hapus Transaksi")
        st.caption("Masukkan ID transaksi yang ingin dihapus (lihat kolom id pada tabel di atas).")
        col_hapus1, _ = st.columns([1, 2])
        with col_hapus1:
            id_hapus = st.number_input("ID Transaksi yang Ingin Dihapus:", min_value=1,
                                       step=1, value=None, placeholder="Contoh: 3", key="input_id_hapus")
        if id_hapus is not None:
            id_hapus_int = int(id_hapus)
            data_terpilih = df_transaksi[df_transaksi['id'] == id_hapus_int]
            if data_terpilih.empty:
                st.warning(f"Tidak ditemukan transaksi dengan ID {id_hapus_int}.")
            else:
                baris = data_terpilih.iloc[0]
                st.info(f"Preview: ID {baris['id']} | {baris['tanggal']} | "
                        f"{baris['kategori']} | {baris['deskripsi']} | {baris['Jumlah (Rp)']}")
                if 'konfirmasi_hapus' not in st.session_state: st.session_state.konfirmasi_hapus = False
                if 'id_dikonfirmasi' not in st.session_state: st.session_state.id_dikonfirmasi = None
                col_b1, col_b2, _ = st.columns([1, 1, 4])
                with col_b1:
                    if st.button("\U0001f5d1\ufe0f Hapus Transaksi Ini", type="primary", key="btn_hapus_init"):
                        st.session_state.konfirmasi_hapus = True
                        st.session_state.id_dikonfirmasi = id_hapus_int
                if st.session_state.konfirmasi_hapus and st.session_state.id_dikonfirmasi == id_hapus_int:
                    st.warning(f"Yakin hapus ID {id_hapus_int}? Tindakan ini tidak dapat dibatalkan.", icon="\u26a0\ufe0f")
                    col_k1, col_k2, _ = st.columns([1, 1, 4])
                    with col_k1:
                        if st.button("\u2705 Ya, Hapus Sekarang", type="primary", key="btn_konfirm_hapus"):
                            with st.spinner("Menghapus..."):
                                berhasil = anggaran.hapus_transaksi(id_hapus_int)
                            if berhasil:
                                st.success(f"Transaksi ID {id_hapus_int} berhasil dihapus!")
                                st.session_state.konfirmasi_hapus = False
                                st.session_state.id_dikonfirmasi = None
                                st.cache_data.clear(); st.rerun()
                            else:
                                st.error(f"Gagal menghapus transaksi ID {id_hapus_int}.")
                                st.session_state.konfirmasi_hapus = False
                    with col_k2:
                        if st.button("\u274c Batal", key="btn_batal_hapus"):
                            st.session_state.konfirmasi_hapus = False
                            st.session_state.id_dikonfirmasi = None; st.rerun()

def halaman_ringkasan(anggaran):
    st.subheader("\U0001f4ca Ringkasan Pengeluaran")
    col_filter1, col_filter2 = st.columns([1, 2])
    with col_filter1:
        pilihan_periode = st.selectbox("Filter Periode:", ["Semua Waktu", "Hari Ini", "Pilih Tanggal"],
                                       key="filter_periode", on_change=lambda: st.cache_data.clear())
    tanggal_filter = None; label_periode = "(Semua Waktu)"
    if pilihan_periode == "Hari Ini":
        tanggal_filter = datetime.date.today(); label_periode = f"({tanggal_filter.strftime('%d %b')})"
    elif pilihan_periode == "Pilih Tanggal":
        if 'tanggal_pilihan_state' not in st.session_state: st.session_state.tanggal_pilihan_state = datetime.date.today()
        tanggal_filter = st.date_input("Pilih Tanggal:", value=st.session_state.tanggal_pilihan_state, key="tanggal_pilihan",
                                       on_change=lambda: setattr(st.session_state, 'tanggal_pilihan_state',
                                       st.session_state.tanggal_pilihan) or st.cache_data.clear())
        label_periode = f"({tanggal_filter.strftime('%d %b %Y')})"
    with col_filter2:
        @st.cache_data(ttl=300)
        def hitung_total_cached(tgl_filter): return anggaran.hitung_total_pengeluaran(tanggal=tgl_filter)
        total_pengeluaran = hitung_total_cached(tanggal_filter)
        st.metric(label=f"Total Pengeluaran {label_periode}", value=format_rp(total_pengeluaran))
    st.divider()
    st.subheader(f"Pengeluaran per Kategori {label_periode}")
    @st.cache_data(ttl=300)
    def get_kategori_cached(tgl_filter): return anggaran.get_pengeluaran_per_kategori(tanggal=tgl_filter)
    with st.spinner("Memuat ringkasan kategori..."): dict_per_kategori = get_kategori_cached(tanggal_filter)
    if not dict_per_kategori: st.info("Tidak ada data untuk periode ini.")
    else:
        try:
            import pandas as pd
            data_kategori = [{"Kategori": kat, "Total": jml} for kat, jml in dict_per_kategori.items()]
            df_kategori = pd.DataFrame(data_kategori).sort_values(by="Total", ascending=False).reset_index(drop=True)
            df_kategori['Total (Rp)'] = df_kategori['Total'].apply(format_rp)
            col_kat1, col_kat2 = st.columns(2)
            with col_kat1: st.write("Tabel:"); st.dataframe(df_kategori[['Kategori', 'Total (Rp)']], hide_index=True, use_container_width=True)
            with col_kat2: st.write("Grafik:"); st.bar_chart(df_kategori.set_index('Kategori')['Total'], use_container_width=True)
        except Exception as e: st.error(f"Gagal menampilkan ringkasan: {e}")

def main():
    st.sidebar.title("\U0001fa99 Catatan Pengeluaran")
    menu_pilihan = st.sidebar.radio("Pilih Menu:", ["Tambah", "Riwayat", "Ringkasan"], key="menu_utama")
    st.sidebar.markdown("---")
    st.sidebar.info("Jobsheet - Aplikasi Keuangan")
    manajer_anggaran = get_anggaran_manager()
    if menu_pilihan == "Tambah": halaman_input(manajer_anggaran)
    elif menu_pilihan == "Riwayat": halaman_riwayat(manajer_anggaran)
    elif menu_pilihan == "Ringkasan": halaman_ringkasan(manajer_anggaran)
    st.markdown("---")
    st.caption("Pengembangan Aplikasi Berbasis OOP")

if __name__ == "__main__":
    main()
"""

# Simpan ke file main_app.py
with open("main_app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code.strip())

print("[OK] main_app.py berhasil disimpan.")
print("     Jalankan dengan: streamlit run main_app.py")

[OK] main_app.py berhasil disimpan.
     Jalankan dengan: streamlit run main_app.py


### Langkah 6: Menjalankan dan Menguji Aplikasi Modular

In [ ]:
#LANGKAH 6: Pengujian Terintegrasi Semua Modul

# Reset database untuk pengujian bersih
import os
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Database lama dihapus untuk demo bersih.")

AnggaranHarian._db_setup_done = False
anggaran = AnggaranHarian()

# --- Uji tambah data valid ---
print("\n=== Uji Tambah Data Valid ===")
data_uji = [
    ("Makan siang",    25000,  "Makanan",       datetime.date.today()),
    ("Ojek online",    15000,  "Transportasi",  datetime.date.today()),
    ("Beli buku OOP",  45000,  "Pendidikan",    datetime.date.today()),
    ("Bayar listrik", 150000,  "Tagihan",       datetime.date.today()),
    ("Nonton bioskop", 60000,  "Hiburan",       datetime.date.today()),
]
for deskripsi, jumlah, kategori, tanggal in data_uji:
    tx = Transaksi(deskripsi, jumlah, kategori, tanggal)
    hasil = anggaran.tambah_transaksi(tx)
    print(f"  Tambah '{deskripsi}': {'OK' if hasil else 'GAGAL'}")

# --- Uji tambah data invalid ---
print("\n=== Uji Tambah Data Tidak Valid ===")
tx_invalid = Transaksi("Uji invalid", -999, "Lainnya", datetime.date.today())
print(f"  Tambah jumlah negatif: {'OK' if anggaran.tambah_transaksi(tx_invalid) else 'Ditolak (sesuai harapan)'}")

# --- Uji lihat riwayat ---
print("\n=== Uji Lihat Riwayat ===")
df = anggaran.get_dataframe_transaksi()
print(df.to_string(index=False))

# --- Uji hitung total ---
print(f"\n=== Uji Hitung Total ===")
total = anggaran.hitung_total_pengeluaran()
print(f"  Total pengeluaran: Rp {total:,.0f}")

# --- Uji ringkasan per kategori ---
print("\n=== Uji Ringkasan Kategori ===")
per_kat = anggaran.get_pengeluaran_per_kategori()
for kat, jml in per_kat.items():
    print(f"  {kat:<15}: Rp {jml:,.0f}")

# --- Uji filter tanggal ---
print("\n=== Uji Filter Tanggal Hari Ini ===")
total_hari_ini = anggaran.hitung_total_pengeluaran(tanggal=datetime.date.today())
print(f"  Total hari ini: Rp {total_hari_ini:,.0f}")

Database lama dihapus untuk demo bersih.
[AnggaranHarian] Melakukan pengecekan/setup database awal...
Memeriksa/membuat tabel di database (via database.py): /content/pengeluaran_harian.db
 -> Tabel 'transaksi' siap.
[AnggaranHarian] Database siap.

=== Uji Tambah Data Valid ===
  Tambah 'Makan siang': OK
  Tambah 'Ojek online': OK
  Tambah 'Beli buku OOP': OK
  Tambah 'Bayar listrik': OK
  Tambah 'Nonton bioskop': OK

=== Uji Tambah Data Tidak Valid ===
Peringatan: Jumlah '-999' harus positif.
  Tambah jumlah negatif: Ditolak (sesuai harapan)

=== Uji Lihat Riwayat ===
 id    tanggal     kategori      deskripsi Jumlah (Rp)
  5 2026-06-05      Hiburan Nonton bioskop   Rp 60.000
  4 2026-06-05      Tagihan  Bayar listrik  Rp 150.000
  3 2026-06-05   Pendidikan  Beli buku OOP   Rp 45.000
  2 2026-06-05 Transportasi    Ojek online   Rp 15.000
  1 2026-06-05      Makanan    Makan siang   Rp 25.000

=== Uji Hitung Total ===
  Total pengeluaran: Rp 295,000

=== Uji Ringkasan Kategori ===
  Ta

### PENUGASAN: Fungsionalitas Hapus Transaksi

In [ ]:
# PENUGASAN: Tambahkan metode hapus_transaksi pada kelas AnggaranHarian

# ── Backend: Metode hapus_transaksi ──────────────────────────────────────
def _hapus_transaksi(self, id_transaksi: int) -> bool:
    """
    Menghapus transaksi berdasarkan ID dari database.

    Args:
        id_transaksi (int): ID transaksi yang ingin dihapus.

    Returns:
        bool: True jika penghapusan berhasil, False jika gagal.
    """
    if not isinstance(id_transaksi, int) or id_transaksi <= 0:
        print(f"[AnggaranHarian] ID transaksi tidak valid: {id_transaksi}")
        return False

    sql = "DELETE FROM transaksi WHERE id = ?"
    params = (id_transaksi,)
    result = execute_query(sql, params)

    # execute_query mengembalikan None hanya jika terjadi error SQLite
    if result is not None:
        print(f"[AnggaranHarian] Transaksi ID {id_transaksi} berhasil dihapus.")
        return True
    else:
        print(f"[AnggaranHarian] Gagal menghapus transaksi ID {id_transaksi}.")
        return False

# Pasang metode ke kelas AnggaranHarian
AnggaranHarian.hapus_transaksi = _hapus_transaksi

print("[OK] Metode hapus_transaksi berhasil ditambahkan ke AnggaranHarian.")

[OK] Metode hapus_transaksi berhasil ditambahkan ke AnggaranHarian.


In [ ]:
# ── Demo & Pengujian Fitur Hapus Transaksi ───────────────────────────────

GARIS = "=" * 60

print(GARIS)
print("  DEMO PENUGASAN: Hapus Transaksi")
print(GARIS)

# Tampilkan data sebelum dihapus
print("\nData sebelum penghapusan:")
df_sebelum = anggaran.get_dataframe_transaksi()
print(df_sebelum.to_string(index=False))
print(f"\nJumlah transaksi: {len(df_sebelum)}")

# ── Uji 1: Hapus transaksi ID 2 (Ojek online) ─────────────────────────
print(f"\n{GARIS}")
print("  UJI 1: Hapus Transaksi ID 2 (Ojek online)")
print(GARIS)
berhasil_1 = anggaran.hapus_transaksi(2)
print(f"Hasil hapus ID 2: {'BERHASIL ✅' if berhasil_1 else 'GAGAL ❌'}")

# ── Uji 2: Hapus transaksi ID 4 (Bayar listrik) ───────────────────────
print(f"\n{GARIS}")
print("  UJI 2: Hapus Transaksi ID 4 (Bayar listrik)")
print(GARIS)
berhasil_2 = anggaran.hapus_transaksi(4)
print(f"Hasil hapus ID 4: {'BERHASIL ✅' if berhasil_2 else 'GAGAL ❌'}")

# ── Uji 3: Hapus ID yang tidak ada ────────────────────────────────────
print(f"\n{GARIS}")
print("  UJI 3: Hapus ID Tidak Ada (ID=9999)")
print(GARIS)
berhasil_3 = anggaran.hapus_transaksi(9999)
print(f"Hasil hapus ID 9999: {'BERHASIL (query jalan tanpa error SQLite)' if berhasil_3 else 'GAGAL ❌'}")

# ── Uji 4: Hapus dengan ID tidak valid ────────────────────────────────
print(f"\n{GARIS}")
print("  UJI 4: Hapus dengan ID Tidak Valid (ID=-1)")
print(GARIS)
berhasil_4 = anggaran.hapus_transaksi(-1)
print(f"Hasil hapus ID -1: {'BERHASIL' if berhasil_4 else 'Ditolak (sesuai harapan) ✅'}")

# ── Tampilkan data setelah semua penghapusan ──────────────────────────
print(f"\n{GARIS}")
print("  Data setelah penghapusan:")
print(GARIS)
df_sesudah = anggaran.get_dataframe_transaksi()
print(df_sesudah.to_string(index=False))
print(f"\nJumlah transaksi setelah hapus : {len(df_sesudah)} (sebelumnya: {len(df_sebelum)})")
print(f"Total pengeluaran setelah hapus : Rp {anggaran.hitung_total_pengeluaran():,.0f}")

print(f"\n{GARIS}")
print("  RINGKASAN KATEGORI SETELAH HAPUS")
print(GARIS)
for kat, jml in anggaran.get_pengeluaran_per_kategori().items():
    print(f"  {kat:<15}: Rp {jml:,.0f}")

print(f"\n{GARIS}")
print("  SEMUA PENGUJIAN SELESAI ✅")
print(GARIS)

  DEMO PENUGASAN: Hapus Transaksi

Data sebelum penghapusan:
 id    tanggal     kategori      deskripsi Jumlah (Rp)
  5 2026-06-05      Hiburan Nonton bioskop   Rp 60.000
  4 2026-06-05      Tagihan  Bayar listrik  Rp 150.000
  3 2026-06-05   Pendidikan  Beli buku OOP   Rp 45.000
  2 2026-06-05 Transportasi    Ojek online   Rp 15.000
  1 2026-06-05      Makanan    Makan siang   Rp 25.000

Jumlah transaksi: 5

  UJI 1: Hapus Transaksi ID 2 (Ojek online)
[AnggaranHarian] Transaksi ID 2 berhasil dihapus.
Hasil hapus ID 2: BERHASIL ✅

  UJI 2: Hapus Transaksi ID 4 (Bayar listrik)
[AnggaranHarian] Transaksi ID 4 berhasil dihapus.
Hasil hapus ID 4: BERHASIL ✅

  UJI 3: Hapus ID Tidak Ada (ID=9999)
[AnggaranHarian] Transaksi ID 9999 berhasil dihapus.
Hasil hapus ID 9999: BERHASIL (query jalan tanpa error SQLite)

  UJI 4: Hapus dengan ID Tidak Valid (ID=-1)
[AnggaranHarian] ID transaksi tidak valid: -1
Hasil hapus ID -1: Ditolak (sesuai harapan) ✅

  Data setelah penghapusan:
 id    tanggal  